In [1]:
import os 
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
MLFLOW_TRACKING_URI = os.getenv('MLFLOW_TRACKING_URI')
MLFLOW_TRACKING_USERNAME = os.getenv('MLFLOW_TRACKING_USERNAME')
MLFLOW_TRACKING_PASSWORD = os.getenv('MLFLOW_TRACKING_PASSWORD')

In [3]:
%pwd

'/home/ammar/Documents/Learning/MLOps/DS_PROJECT/research'

In [4]:
os.chdir('../')
%pwd

'/home/ammar/Documents/Learning/MLOps/DS_PROJECT'

In [5]:
from dataclasses import dataclass
from pathlib import Path


@dataclass
class ModelEvaluationConfig:
    root_dir: Path
    test_data_path: Path
    model_path: Path
    all_params: dict
    metric_file_name: Path
    target_column: str
    mlflow_uri: str


In [6]:
from src.DS_PROJECT.constants import *
from src.DS_PROJECT.utils.common import read_yaml,create_directories

In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config=self.config.model_evaluation
        params=self.params.ElasticNet
        schema=self.schema.TARGET_COLUMN

        create_directories([config.root_dir])

        model_evaluation_config=ModelEvaluationConfig(
            root_dir=config.root_dir,
            test_data_path=config.test_data_path,
            model_path = config.model_path,
            all_params=params,
            metric_file_name = config.metric_file_name,
            target_column = schema.name,
            mlflow_uri=MLFLOW_TRACKING_URI


        )
        return model_evaluation_config


In [8]:
import os
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from urllib.parse import urlparse
import mlflow
import mlflow.sklearn
from mlflow.exceptions import RestException
import numpy as np
import joblib
import json

def save_json(path, data):
    with open(path, "w") as f:
        json.dump(data, f, indent=4)


In [ ]:
class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config

    def eval_metrics(self,actual, pred):
        rmse = np.sqrt(mean_squared_error(actual, pred))
        mae = mean_absolute_error(actual, pred)
        r2 = r2_score(actual, pred)
        return rmse, mae, r2
    
    def log_into_mlflow(self):

        test_data = pd.read_csv(self.config.test_data_path)
        model = joblib.load(self.config.model_path)

        test_x = test_data.drop([self.config.target_column], axis=1)
        test_y = test_data[[self.config.target_column]]


        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme

        with mlflow.start_run():

            predicted_qualities = model.predict(test_x)

            (rmse, mae, r2) = self.eval_metrics(test_y, predicted_qualities)
            
            # Saving metrics as local
            scores = {"rmse": rmse, "mae": mae, "r2": r2}
            save_json(path=Path(self.config.metric_file_name), data=scores)

            mlflow.log_params(self.config.all_params)

            mlflow.log_metric("rmse", rmse)
            mlflow.log_metric("r2", r2)
            mlflow.log_metric("mae", mae)


            # Log the model (handle DagsHub limitation with model registry endpoints)
            try:
                mlflow.sklearn.log_model(model, "model")
            except RestException as e:
                # DagsHub doesn't support the logged model endpoint used by newer MLflow versions
                # Fall back to logging model as a regular artifact
                import tempfile
                
                with tempfile.TemporaryDirectory() as tmpdir:
                    model_path = os.path.join(tmpdir, "model.joblib")
                    joblib.dump(model, model_path)
                    # Log as artifact
                    mlflow.log_artifact(model_path, "model")
                print(f"Note: Model logged as artifact (backend limitation: unsupported endpoint)")
            except Exception as e:
                # Re-raise if it's a different error
                raise e
    


In [10]:
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation = ModelEvaluation(config=model_evaluation_config)
    model_evaluation.log_into_mlflow()
except Exception as e:
    raise e

[2025-12-06 09:17:34,798]: INFO : common : yaml file: config/config.yaml loaded successfully:
[2025-12-06 09:17:34,799]: INFO : common : yaml file: params.yaml loaded successfully:
[2025-12-06 09:17:34,801]: INFO : common : yaml file: schema.yaml loaded successfully:
[2025-12-06 09:17:34,801]: INFO : common : created directory at: artifacts:
[2025-12-06 09:17:34,801]: INFO : common : created directory at: artifacts/model_evaluation:


2025/12/06 09:17:36 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Note: Model logged as artifact (backend limitation: unsupported endpoint)
🏃 View run fortunate-squid-795 at: https://dagshub.com/Ammar.ai/DS_Project.mlflow/#/experiments/0/runs/bd304279398c435fb45db49309afd1b8
🧪 View experiment at: https://dagshub.com/Ammar.ai/DS_Project.mlflow/#/experiments/0


In [11]:
!pip install --upgrade mlflow
